# A07_validate: how right are the rules?

| Step | Purpose |
|---|---|
| 7.1 | Sample 20 rows per residence `kind` × `precision`, and 20 per reason code |
| 7.2 | Label each by reading its snippet: correct / wrong place / wrong kind / unclear |
| 7.3 | Precision = correct ÷ labelled; coverage = files with ≥1 row ÷ files with a declaration |
| 7.4 | Which rules to keep, fix or drop |

In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote
import pandas as pd

## 7.1 Sample
Shuffle once, then keep the first 20 rows of each group (pandas 3 drops group columns inside `groupby().apply`). Written only if no labelled file exists, so re-running never wipes the labels.

In [2]:
res = io.read_stage("interim", "residences")
rea = io.read_stage("interim", "reasons")
VPATH = config.INTERIM_DIR / "validation.csv"

s1 = (res[res.method == "declaration_regex"].sample(frac=1, random_state=11)
        .groupby(["kind", "precision"]).head(20))
s1 = s1.assign(table="residences", kind_or_code=s1.kind, place_or_snippet=s1.place_id + " | " + s1.snippet)
s2 = rea.sample(frac=1, random_state=11).groupby("reason_code").head(20)
s2 = s2.assign(table="reasons", kind_or_code=s2.reason_code, place_or_snippet=s2.snippet.str[:250])
sample = pd.concat([s1, s2])[["table", "NAID", "pageNumber", "kind_or_code", "place_or_snippet"]].assign(label="", note="")

if VPATH.exists() and pd.read_csv(VPATH).label.notna().any():
    print("labels already in", VPATH.name, "- not overwriting")
else:
    sample.to_csv(VPATH, index=False)
sample.groupby(["table", "kind_or_code"]).size()

labels already in validation.csv - not overwriting


table       kind_or_code 
reasons     age_health       20
            climate          20
            conflict         20
            family           20
            health           20
            land             20
            land_work        20
            pension_admin    20
            poverty          20
            spouse           20
            war_damage       20
residences  born             40
            enlisted         40
            lived_since      40
            married          40
            now_resides      40
            removed_to       40
dtype: int64

## 7.2 Labels
Labelled by Claude on 2026-11-22, at Lara's request, by reading each snippet (column `labeller`).

- **correct:** the place (at its precision) and kind match what the text says; for reasons, the code is a reason given for *this* move
- **wrong place:** a different place than the one stated (e.g. Antrim, Ireland → Antrim Co., Michigan)
- **wrong kind:** a real place, but not this kind (a witness's town, the residence at enlistment tagged `now_resides`), or a keyword that is not a reason
- **unclear:** the snippet alone can't tell ("said county", "same county & state")

`note` flags state-level rows where a county was stated but not captured.

In [3]:
v = pd.read_csv(VPATH)
assert v.label.notna().all()
pd.crosstab(v.kind_or_code, v.label, margins=True)

label,correct,unclear,wrong kind,wrong place,All
kind_or_code,,,,,
age_health,18,1,1,0,20
born,35,1,1,3,40
climate,16,3,1,0,20
conflict,0,1,19,0,20
enlisted,35,2,0,3,40
family,18,1,1,0,20
health,4,0,16,0,20
land,9,5,6,0,20
land_work,18,0,2,0,20


## 7.3 Precision and coverage
Precision two ways: strict (unclear counts as wrong) and on decided rows only (unclear left out).

In [4]:
def prec(g):
    dec = g[g.label != "unclear"]
    return pd.Series({"n": len(g), "strict": (g.label == "correct").mean(),
                      "decided": (dec.label == "correct").mean(),
                      "county_missed": g.note.fillna("").str.contains("county stated").sum()})
p = v.groupby(["table", "kind_or_code"]).apply(prec, include_groups=False).round(2)
p["keep"] = p.decided.ge(0.8).map({True: "keep", False: "fix / drop"})
p

n  strict  decided  county_missed        keep
table      kind_or_code                                                   
reasons    age_health     20.0    0.90     0.95            0.0        keep
           climate        20.0    0.80     0.94            0.0        keep
           conflict       20.0    0.00     0.00            0.0  fix / drop
           family         20.0    0.90     0.95            0.0        keep
           health         20.0    0.20     0.20            0.0  fix / drop
           land           20.0    0.45     0.60            0.0  fix / drop
           land_work      20.0    0.90     0.90            0.0        keep
           pension_admin  20.0    0.15     0.15            0.0  fix / drop
           poverty        20.0    0.75     0.75            0.0  fix / drop
           spouse         20.0    0.50     0.53            0.0  fix / drop
           war_damage     20.0    0.10     0.11            0.0  fix / drop
residences born           40.0    0.88     0.90            3.0        keep
           enlisted       40.0    0.88     0.92            5.0        keep
           lived_since    40.0    0.78     0.91            1.0        keep
           married        40.0    0.80     0.94            1.0        keep
           now_resides    40.0    0.80     0.89            5.0        keep
           removed_to     40.0    0.92     0.97            8.0        keep

In [5]:
decl = io.read_stage("interim", "decl_text", columns=["NAID"]).NAID.unique()
files = io.read_stage("interim", "files", columns=["NAID", "ftype"])
rp = io.read_stage("interim", "reason_pages", columns=["NAID"]).NAID.unique()
regex_files = res.loc[res.method == "declaration_regex", "NAID"].unique()
now_files = res.loc[(res.method == "declaration_regex") & (res.kind == "now_resides"), "NAID"].unique()
cov = pd.Series({
    "files": len(files),
    "files with a declaration": len(decl),
    "≥1 residence row (regex)": len(regex_files),
    "a now_resides row (regex)": len(now_files),
    "any residence incl. fallback (also files with no declaration)": res.NAID.nunique(),
    "reason_pages files": len(rp),
    "files with a coded reason statement": rea.loc[rea.source == "reason_statement", "NAID"].nunique(),
})
cov.to_frame("n").assign(pct_of_decl=(cov / len(decl) * 100).round(1))

,n,pct_of_decl
files,78745,124.9
files with a declaration,63030,100.0
≥1 residence row (regex),56402,89.5
a now_resides row (regex),54954,87.2
any residence incl. fallback (also files with no declaration),64315,102.0
reason_pages files,2079,3.3
files with a coded reason statement,1638,2.6


In [6]:
# residence precision weighted by how many rows each kind × precision really has
w = res[res.method == "declaration_regex"].groupby(["kind", "precision"]).size().rename("rows")
lab = v[v.table == "residences"].assign(kind=s1.kind.values, precision=s1.precision.values)   # same seed → same row order
assert (lab.kind == lab.kind_or_code).all()
pp = lab.groupby(["kind", "precision"]).label.apply(lambda s: (s == "correct").mean()).rename("strict")
t = pd.concat([w, pp], axis=1)
print("row-weighted strict precision, residences:", round((t.rows * t.strict).sum() / t.rows.sum(), 3))
t.round(2)

row-weighted strict precision, residences: 0.827


rows  strict
kind        precision               
born        county      6471    0.80
            state       7765    0.95
enlisted    county      2059    0.95
            state       2714    0.80
lived_since county        84    0.90
            state        189    0.65
married     county      2483    0.80
            state       3708    0.80
now_resides county     31335    0.85
            state      34226    0.75
removed_to  county      4714    0.90
            state       6219    0.95

## 7.4 Verdict
**Residences: keep all six rules.** Each scores ≥ 80% on decided rows. Errors repeat in a few shapes, which suggests the fixes for a v2:
- `now_resides` sometimes grabs the residence *at enlistment* ("a resident of Bladen when he enlisted") or a witness certificate. Fix: skip windows that contain "when he enlisted" / "certify".
- `born` / `enlisted` / `removed_to` windows run past the place into a *later* place ("born in Westchester … lived in Fairfield"). Fix: stop the window at the first "and / then / thence / lived / moved".
- County names that exist in several states match the wrong one when no state is given (Antrim, Ireland → Michigan; Greenfield MA → Franklin Co., NY). Fix: only take a unique county name when the window has no foreign word ("ireland", "england").
- About 1 in 5 state-level rows had a county in the text that wasn't captured ("county of madison aged…"). Fix: let the county regex end at "aged / who / in the state".
- "said county / aforesaid" rows are mostly **unclear** from the snippet; the header borrowing is probably right, but it is unverified.

**Reasons: keep the 7 statement codes, drop the declaration-near-move codes from the viz.**
- Statement codes: family, age_health, land_work and climate ≥ 80%. poverty about 75%. **spouse** (about 50%) fires on "his wife is also old" or "daughters who are married". Read it as "spouse mentioned", or tighten it to death/health of a spouse. **pension_admin** fires on "(Pittsburgh agency)" in nearly every transfer. Keep only "convenient / cheaper to draw the pension".
- Declaration codes: `conflict` 0%, `war_damage` 10%, `health` 20%, `land` about 60%. "Moved" in a service narrative is usually troops moving. These need the model gate, or leave them out.

Numbers go to `docs/DATA.md`.